# Setup

In [1]:
import json
import numpy as np
import tensorflow as tf

2025-12-09 10:30:49.441774: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-12-09 10:30:49.442148: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-12-09 10:30:49.444559: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-12-09 10:30:49.451339: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1765287049.462904   99463 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1765287049.46

In [2]:
# Normalization only using training data
tfrecord_file = "../data/train.tfrecord"

# Reading TFRecord file

In [3]:
def parse_fn(example):
    features = {
        # Inputs
        "NOISY_ALBEDO_B-NIR": tf.io.VarLenFeature(tf.float32),
        "NOISY_ALBEDO_B-UV": tf.io.VarLenFeature(tf.float32),
        "NOISY_ALBEDO_B-Vis": tf.io.VarLenFeature(tf.float32),
        "NOISY_ALBEDO_SS-NIR": tf.io.VarLenFeature(tf.float32),
        "NOISY_ALBEDO_SS-UV": tf.io.VarLenFeature(tf.float32),
        "NOISY_ALBEDO_SS-Vis": tf.io.VarLenFeature(tf.float32),
        # Planetary params
        "OBJECT-RADIUS-REL-EARTH": tf.io.FixedLenFeature([], tf.float32),
        "OBJECT-GRAVITY": tf.io.FixedLenFeature([], tf.float32),
        "ATMOSPHERE-TEMPERATURE": tf.io.FixedLenFeature([], tf.float32),
        "ATMOSPHERE-PRESSURE": tf.io.FixedLenFeature([], tf.float32),
        # Chemical abundances
        # 'C2H6': tf.io.FixedLenFeature([], tf.float32),
        "CH4": tf.io.FixedLenFeature([], tf.float32),
        #'CO':   tf.io.FixedLenFeature([], tf.float32),
        "CO2": tf.io.FixedLenFeature([], tf.float32),
        "H2O": tf.io.FixedLenFeature([], tf.float32),
        "N2": tf.io.FixedLenFeature([], tf.float32),
        #'N2O':  tf.io.FixedLenFeature([], tf.float32),
        "O2": tf.io.FixedLenFeature([], tf.float32),
        "O3": tf.io.FixedLenFeature([], tf.float32),
    }

    parsed_features = tf.io.parse_single_example(example, features)

    dense_features = {
        key: (
            tf.sparse.to_dense(value, default_value=0.0)
            if isinstance(value, tf.SparseTensor)
            else value
        )
        for key, value in parsed_features.items()
    }

    return dense_features


dataset = tf.data.TFRecordDataset(tfrecord_file)
dataset = dataset.map(parse_fn)

2025-12-09 10:31:07.080348: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [4]:
# Take this information from the previous notebook (07-hellinger_transform.ipynb)
best_n_values = {
    "CH4": np.float64(6.75),
    "CO2": np.float64(3.7),
    "H2O": np.float64(5.6000000000000005),
    "N2": np.float64(0.4),
    "O2": np.float64(2.35),
    "O3": np.float64(13.750000000000002),
}

# Normalization statistics into a dictionary

In [6]:
def compute_normalization_stats(train_tfrecord_path, best_n_values):
    stats = {
        "inputs": {
            "B-UV": {"sum": 0.0, "sq_sum": 0.0, "count": 0},
            "B-Vis": {"sum": 0.0, "sq_sum": 0.0, "count": 0},
            "B-NIR": {"sum": 0.0, "sq_sum": 0.0, "count": 0},
            "SS-UV": {"sum": 0.0, "sq_sum": 0.0, "count": 0},
            "SS-Vis": {"sum": 0.0, "sq_sum": 0.0, "count": 0},
            "SS-NIR": {"sum": 0.0, "sq_sum": 0.0, "count": 0},
        },
        "outputs": {
            # Planetary parameters
            "OBJECT-RADIUS-REL-EARTH": {"min": np.inf, "max": -np.inf, "best_n": 1},
            "OBJECT-GRAVITY": {"min": np.inf, "max": -np.inf, "best_n": 1},
            "ATMOSPHERE-TEMPERATURE": {"min": np.inf, "max": -np.inf, "best_n": 1},
            "ATMOSPHERE-PRESSURE": {"min": np.inf, "max": -np.inf, "best_n": 1},
            # Chemical abundances
            # 'C2H6': {'sum': 0., 'sq_sum': 0., 'count': 0},
            "CH4": {"best_n": 1},
            #'CO':   {'best_n' : 1},
            "CO2": {"best_n": 1},
            "H2O": {"best_n": 1},
            "N2": {"best_n": 1},
            # 'N2O':  {'best_n' : 1},
            "O2": {"best_n": 1},
            "O3": {"best_n": 1},
        },
    }

    def parse_fn(example):
        features = {
            # Inputs
            "NOISY_ALBEDO_B-NIR": tf.io.VarLenFeature(tf.float32),
            "NOISY_ALBEDO_B-UV": tf.io.VarLenFeature(tf.float32),
            "NOISY_ALBEDO_B-Vis": tf.io.VarLenFeature(tf.float32),
            "NOISY_ALBEDO_SS-NIR": tf.io.VarLenFeature(tf.float32),
            "NOISY_ALBEDO_SS-UV": tf.io.VarLenFeature(tf.float32),
            "NOISY_ALBEDO_SS-Vis": tf.io.VarLenFeature(tf.float32),
            # Planetary params
            "OBJECT-RADIUS-REL-EARTH": tf.io.FixedLenFeature([], tf.float32),
            "OBJECT-GRAVITY": tf.io.FixedLenFeature([], tf.float32),
            "ATMOSPHERE-TEMPERATURE": tf.io.FixedLenFeature([], tf.float32),
            "ATMOSPHERE-PRESSURE": tf.io.FixedLenFeature([], tf.float32),
            # Chemical abundances
            # 'C2H6': tf.io.FixedLenFeature([], tf.float32),
            "CH4": tf.io.FixedLenFeature([], tf.float32),
            #'CO':   tf.io.FixedLenFeature([], tf.float32),
            "CO2": tf.io.FixedLenFeature([], tf.float32),
            "H2O": tf.io.FixedLenFeature([], tf.float32),
            "N2": tf.io.FixedLenFeature([], tf.float32),
            #'N2O':  tf.io.FixedLenFeature([], tf.float32),
            "O2": tf.io.FixedLenFeature([], tf.float32),
            "O3": tf.io.FixedLenFeature([], tf.float32),
        }
        return tf.io.parse_single_example(example, features)

    dataset = tf.data.TFRecordDataset(train_tfrecord_path)
    dataset = dataset.map(parse_fn)

    for batch in dataset.batch(1000):  # Process in chunks
        for region in ["B-UV", "B-Vis", "B-NIR", "SS-UV", "SS-Vis", "SS-NIR"]:
            key = f"NOISY_ALBEDO_{region}"
            data = tf.sparse.to_dense(batch[key]).numpy()

            stats["inputs"][region]["sum"] += np.sum(data)
            stats["inputs"][region]["sq_sum"] += np.sum(data**2)
            stats["inputs"][region]["count"] += data.size

        for param in [
            "OBJECT-RADIUS-REL-EARTH",
            "OBJECT-GRAVITY",
            "ATMOSPHERE-TEMPERATURE",
            "ATMOSPHERE-PRESSURE",
        ]:
            data = batch[param].numpy()
            stats["outputs"][param]["min"] = min(
                stats["outputs"][param]["min"], np.min(data)
            )
            stats["outputs"][param]["max"] = max(
                stats["outputs"][param]["max"], np.max(data)
            )

        for chem in ["CH4", "CO2", "H2O", "N2", "O2", "O3"]:
            data = batch[chem].numpy()
            stats["outputs"][chem]["best_n"] = best_n_values[chem]

    final_stats = {"inputs": {}, "outputs": {}}

    for region in ["B-UV", "B-Vis", "B-NIR", "SS-UV", "SS-Vis", "SS-NIR"]:
        s = stats["inputs"][region]["sum"]
        sq_s = stats["inputs"][region]["sq_sum"]
        cnt = stats["inputs"][region]["count"]

        mean = s / cnt
        var = (sq_s / cnt) - (mean**2)
        std = np.sqrt(var)

        final_stats["inputs"][region] = {"mean": float(mean), "std": float(std)}

    for param in [
        "OBJECT-RADIUS-REL-EARTH",
        "OBJECT-GRAVITY",
        "ATMOSPHERE-TEMPERATURE",
        "ATMOSPHERE-PRESSURE",
    ]:
        min_ = stats["outputs"][param]["min"]
        max_ = stats["outputs"][param]["max"]
        best_n = stats["outputs"][param]["best_n"]

        final_stats["outputs"][param] = {
            "min": float(min_),
            "max": float(max_),
            "best_n": float(best_n),
        }

    for chem in ["CH4", "CO2", "H2O", "N2", "O2", "O3"]:
        best_n = stats["outputs"][chem]["best_n"]
        final_stats["outputs"][chem] = {"best_n": float(best_n)}

    with open("../data/normalization_stats.json", "w") as f:
        json.dump(final_stats, f)

    return final_stats

In [7]:
compute_normalization_stats(tfrecord_file, best_n_values)

2025-12-09 10:33:48.770711: I tensorflow/core/kernels/data/tf_record_dataset_op.cc:381] TFRecordDataset `buffer_size` is unspecified, default to 262144
2025-12-09 10:33:50.317470: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


{'inputs': {'B-UV': {'mean': 0.07439041137695312, 'std': 0.062298085540533066},
  'B-Vis': {'mean': 0.0613805390894413, 'std': 0.04356507956981659},
  'B-NIR': {'mean': 0.032403964549303055, 'std': 0.04065250977873802},
  'SS-UV': {'mean': 0.07432185113430023, 'std': 0.06412459909915924},
  'SS-Vis': {'mean': 0.06520719826221466, 'std': 0.044040221720933914},
  'SS-NIR': {'mean': 0.03764934092760086, 'std': 0.04127572849392891}},
 'outputs': {'OBJECT-RADIUS-REL-EARTH': {'min': 0.6410925984382629,
   'max': 1.2299991846084595,
   'best_n': 1.0},
  'OBJECT-GRAVITY': {'min': 4.704051494598389,
   'max': 13.929901123046875,
   'best_n': 1.0},
  'ATMOSPHERE-TEMPERATURE': {'min': 273.1501770019531,
   'max': 383.092041015625,
   'best_n': 1.0},
  'ATMOSPHERE-PRESSURE': {'min': 234.0108184814453,
   'max': 2043.081298828125,
   'best_n': 1.0},
  'CH4': {'best_n': 6.75},
  'CO2': {'best_n': 3.7},
  'H2O': {'best_n': 5.6000000000000005},
  'N2': {'best_n': 0.4},
  'O2': {'best_n': 2.35},
  'O3'